# Kiswahili cha Mtaani — Data Prep & Fine-Tune Notebook

Notebook hii inafanya kazi pamoja na **Kiswahili cha Mtaani Dataset App** (Next.js + PostgreSQL, iliyowekwa Railway).

Hatua tutakazopitia:
1. Kuunganisha na app yako na kupakua dataset (train/val/test)
2. EDA (Exploratory Data Analysis)
3. Data cleaning + ukaguzi wa data leakage kati ya splits
4. Kuandaa data kwa muundo tayari wa fine-tuning
5. Fine-tuning starter (LoRA) — skeleton ya kuanzia

> **Kumbuka:** Sehemu ya (5) ni *starting point* tu — utahitaji kuchagua base model, kupata
> compute inayolingana (Colab free tier GPU ni ndogo kwa models kubwa), na kurekebisha
> hyperparameters. Sehemu za (1)-(4) ziko tayari kutumika moja kwa moja.

In [ ]:
!pip install -q pandas matplotlib requests datasets

## 1. Kuunganisha na App na Kupakua Dataset

Chagua njia moja:
- **(A) Moja kwa moja kutoka Railway app** — inahitaji APP_URL na APP_PASSWORD
- **(B) Kupakia faili za JSONL ulizopakua mwenyewe** kutoka `/admin` dashboard

In [ ]:
# --- (A) Moja kwa moja kutoka kwenye app yako ---
import requests

APP_URL = "https://JINA-LA-APP-YAKO.up.railway.app"  # badilisha na URL yako halisi
APP_PASSWORD = "weka-password-yako-hapa"

def login_and_get_session(app_url, password):
    session = requests.Session()
    res = session.post(f"{app_url}/api/auth", json={"password": password})
    res.raise_for_status()
    data = res.json()
    if not data.get("ok"):
        raise RuntimeError(f"Login imeshindwa: {data.get('error')}")
    return session

def fetch_jsonl(session, app_url, part="all", train_ratio=0.8, val_ratio=0.1, seed=42, only_good=False):
    params = {
        "format": "jsonl",
        "part": part,
        "trainRatio": train_ratio,
        "valRatio": val_ratio,
        "seed": seed,
        "onlyGood": str(only_good).lower(),
    }
    res = session.get(f"{app_url}/api/export", params=params)
    res.raise_for_status()
    lines = [l for l in res.text.split("\n") if l.strip()]
    return [json.loads(l) for l in lines]

import json
session = login_and_get_session(APP_URL, APP_PASSWORD)
train_records = fetch_jsonl(session, APP_URL, part="train")
val_records = fetch_jsonl(session, APP_URL, part="val")
test_records = fetch_jsonl(session, APP_URL, part="test")
print(f"Train: {len(train_records)} | Val: {len(val_records)} | Test: {len(test_records)}")

In [ ]:
# --- (B) Mbadala: pakia faili za JSONL ulizopakua kutoka /admin ---
# Ondoa # kwenye mistari hii kama unataka kutumia njia hii badala ya (A)

# from google.colab import files
# import json
#
# def load_jsonl_upload():
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     with open(fname, "r", encoding="utf-8") as f:
#         return [json.loads(l) for l in f if l.strip()]
#
# print("Pakia train.jsonl:")
# train_records = load_jsonl_upload()
# print("Pakia val.jsonl:")
# val_records = load_jsonl_upload()
# print("Pakia test.jsonl:")
# test_records = load_jsonl_upload()

In [ ]:
import pandas as pd

train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"--- {name} ---")
    print(df.shape)

train_df.head()

## 2. EDA (Exploratory Data Analysis)

Tunachanganya train+val+test kwa uchambuzi wa jumla (hii ni kuangalia dataset
nzima, si kwa ajili ya kutraining).

In [ ]:
full_df = pd.concat([train_df.assign(split="train"),
                      val_df.assign(split="val"),
                      test_df.assign(split="test")], ignore_index=True)

print("Jumla ya entries:", len(full_df))
print("Jumla ya maneno:", full_df["word_count"].sum())
print()
print(full_df["type"].value_counts())
print()
print(full_df["quality_flag"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

full_df["word_count"].plot(kind="hist", bins=30, ax=axes[0,0], title="Mgawanyo wa urefu wa entries (maneno)")
full_df["region"].value_counts().plot(kind="barh", ax=axes[0,1], title="Idadi ya entries kwa Mkoa")
full_df["topic"].value_counts().plot(kind="barh", ax=axes[1,0], title="Idadi ya entries kwa Mada")
full_df["source_type"].value_counts().plot(kind="barh", ax=axes[1,1], title="Idadi ya entries kwa Chanzo")

plt.tight_layout()
plt.show()

In [ ]:
# Vocabulary / word frequency (whitespace tokenization rahisi)
from collections import Counter
import re

def extract_text(row):
    if row["type"] == "dialogue" and row["turns"]:
        return " ".join(t["text"] for t in row["turns"])
    return row["text"] or ""

full_df["flat_text"] = full_df.apply(extract_text, axis=1)

all_words = []
for t in full_df["flat_text"]:
    words = re.findall(r"[\w']+", t.lower())
    all_words.extend(words)

vocab_counter = Counter(all_words)
print("Vocabulary size (maneno tofauti):", len(vocab_counter))
print("Jumla ya maneno (tokens):", len(all_words))
print()
print("Maneno 30 ya kawaida zaidi:")
for word, count in vocab_counter.most_common(30):
    print(f"  {word}: {count}")

In [ ]:
top_n = 25
top_words = vocab_counter.most_common(top_n)
words, counts = zip(*top_words)

plt.figure(figsize=(10, 6))
plt.barh(words[::-1], counts[::-1])
plt.title(f"Maneno {top_n} ya kawaida zaidi")
plt.tight_layout()
plt.show()

## 3. Data Cleaning + Ukaguzi wa Data Leakage

App yako inafanya deduplication ndani ya dataset nzima, lakini kuna hatari moja
muhimu: **kama entries mbili zinafanana kabisa kabla ya split, shuffle inaweza
kuziweka kwenye splits tofauti** (mfano moja train, nyingine test) — hii ni
*data leakage* na inafanya evaluation yako ionekane bora kuliko ukweli.

Hebu tuangalie hilo kabla ya kuendelea.

In [ ]:
def normalize(t):
    return re.sub(r"\s+", " ", (t or "").strip().lower())

train_keys = set(train_df.apply(extract_text, axis=1).map(normalize))
val_keys = set(val_df.apply(extract_text, axis=1).map(normalize))
test_keys = set(test_df.apply(extract_text, axis=1).map(normalize))

leak_train_val = train_keys & val_keys
leak_train_test = train_keys & test_keys
leak_val_test = val_keys & test_keys

print(f"Leakage train↔val: {len(leak_train_val)}")
print(f"Leakage train↔test: {len(leak_train_test)}")
print(f"Leakage val↔test: {len(leak_val_test)}")

if leak_train_val or leak_train_test or leak_val_test:
    print("\n⚠️ Kuna data leakage — endesha cell inayofuata kuisafisha.")
else:
    print("\n✅ Hakuna leakage kati ya splits.")

In [ ]:
# Kusafisha leakage: tunaweka entry kwenye 'train' na kuifuta kwenye val/test
# (train ndiyo kubwa zaidi kwa kawaida, hivyo hii ndiyo chaguo salama zaidi)

def drop_leaking(df, leak_keys):
    mask = df.apply(extract_text, axis=1).map(normalize).isin(leak_keys)
    removed = mask.sum()
    if removed:
        print(f"Inafuta {removed} entries zinazovuja...")
    return df[~mask].reset_index(drop=True)

val_df = drop_leaking(val_df, leak_train_val | leak_val_test)
test_df = drop_leaking(test_df, leak_train_test | leak_val_test)

print(f"Baada ya kusafisha — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# Kusafisha maandishi: kuondoa whitespace ya ziada, entries fupi mno (chini ya maneno 2)

def clean_df(df, min_words=2):
    df = df.copy()
    df["flat_text"] = df.apply(extract_text, axis=1).map(lambda t: re.sub(r"\s+", " ", t.strip()))
    before = len(df)
    df = df[df["flat_text"].map(lambda t: len(t.split())) >= min_words].reset_index(drop=True)
    after = len(df)
    print(f"Iliondoa {before - after} entries fupi mno (chini ya maneno {min_words}).")
    return df

train_df = clean_df(train_df)
val_df = clean_df(val_df)
test_df = clean_df(test_df)

## 4. Kuandaa Data kwa Fine-Tuning

Tunatengeneza mafomati mawili kutoka kwenye data:

1. **Continued pre-training / plain text** — maandishi ghafi (sentensi na
   mazungumzo yaliyoundwa kama muundo wa mazungumzo) — inafaa kwa kufundisha
   model "kuelewa" mtindo wa lugha
2. **Parallel pairs (sheng ↔ sanifu)** — kwa entries zenye `standard_swahili`,
   tunatengeneza jozi zinazofaa kwa instruction-tuning ya tafsiri

In [ ]:
def format_for_lm(row):
    """Rudisha maandishi ya mafunzo ya causal LM kwa entry moja."""
    if row["type"] == "dialogue" and row["turns"]:
        return "\n".join(f"{t['speaker']}: {t['text']}" for t in row["turns"])
    return row["text"] or ""

def build_lm_dataset(df):
    texts = df.apply(format_for_lm, axis=1)
    return pd.DataFrame({"text": texts})

lm_train = build_lm_dataset(train_df)
lm_val = build_lm_dataset(val_df)
lm_test = build_lm_dataset(test_df)

print(lm_train.shape, lm_val.shape, lm_test.shape)
lm_train.head()

In [ ]:
# Jozi za tafsiri (sheng -> sanifu) kwa instruction-tuning

def build_translation_pairs(df):
    subset = df[(df["type"] == "sentence") & df["standard_swahili"].notna() & (df["standard_swahili"] != "")]
    records = []
    for _, row in subset.iterrows():
        records.append({
            "instruction": "Tafsiri sentensi hii ya Kiswahili cha mtaani kuwa Kiswahili sanifu.",
            "input": row["text"],
            "output": row["standard_swahili"],
        })
    return pd.DataFrame(records)

translation_train = build_translation_pairs(train_df)
translation_val = build_translation_pairs(val_df)

print(f"Jozi za tafsiri — train: {len(translation_train)}, val: {len(translation_val)}")
translation_train.head()

In [ ]:
import os
os.makedirs("prepared_dataset", exist_ok=True)

lm_train.to_json("prepared_dataset/lm_train.jsonl", orient="records", lines=True, force_ascii=False)
lm_val.to_json("prepared_dataset/lm_val.jsonl", orient="records", lines=True, force_ascii=False)
lm_test.to_json("prepared_dataset/lm_test.jsonl", orient="records", lines=True, force_ascii=False)

translation_train.to_json("prepared_dataset/translation_train.jsonl", orient="records", lines=True, force_ascii=False)
translation_val.to_json("prepared_dataset/translation_val.jsonl", orient="records", lines=True, force_ascii=False)

print("Faili zimehifadhiwa kwenye ./prepared_dataset/")
!ls -la prepared_dataset/

In [ ]:
# (Hiari) Kubadilisha kuwa HuggingFace datasets.Dataset — rahisi kutumia na Trainer/SFTTrainer
from datasets import Dataset, DatasetDict

lm_dataset = DatasetDict({
    "train": Dataset.from_pandas(lm_train, preserve_index=False),
    "validation": Dataset.from_pandas(lm_val, preserve_index=False),
    "test": Dataset.from_pandas(lm_test, preserve_index=False),
})

lm_dataset.save_to_disk("prepared_dataset/lm_hf_dataset")
print(lm_dataset)

# Ukitaka kupakia Hugging Face Hub (hiari, inahitaji token):
# from huggingface_hub import login
# login("hf_...")
# lm_dataset.push_to_hub("jina-lako/kiswahili-mtaani")

## 5. Fine-Tuning Starter (LoRA) — Skeleton

Hii ni **msingi wa kuanzia**, siyo pipeline kamili — utahitaji:
- Kuchagua base model inayoruhusu Kiswahili vizuri (mfano models za multilingual
  kama Gemma, Llama 3.x, au models za Kiafrika kama zile za Jacaranda Health/Sartify)
- Kuangalia GPU/VRAM inayopatikana kwenye Colab yako (free tier ina ukomo)
- Kurekebisha hyperparameters (learning rate, batch size, steps) kulingana na
  ukubwa wa dataset yako

Sehemu hii inatumia `peft` (LoRA) + `transformers` — mbinu inayotumika sana kwa
fine-tuning yenye compute ndogo kuliko full fine-tuning.

In [ ]:
!pip install -q peft bitsandbytes accelerate transformers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

# Badilisha hii na model unayotaka kutumia kama msingi
BASE_MODEL = "google/gemma-2-2b"  # mfano — hakikisha una ruhusa/access kwenye HF Hub

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_train = lm_dataset["train"].map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val = lm_dataset["validation"].map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./kiswahili-mtaani-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

# Ondoa # kuanzisha fine-tuning (hakikisha una GPU runtime imechaguliwa: Runtime > Change runtime type)
# trainer.train()

In [ ]:
# Baada ya mafunzo — hifadhi LoRA adapters (ndogo, rahisi kubebeka)
# model.save_pretrained("./kiswahili-mtaani-lora-adapters")
# tokenizer.save_pretrained("./kiswahili-mtaani-lora-adapters")

print("Skeleton tayari. Ondoa # kwenye trainer.train() ukiwa tayari kuanza fine-tuning halisi.")

## Mwisho

Muhtasari wa notebook hii:
- Umepakua data moja kwa moja kutoka kwenye app yako (train/val/test)
- Umeona EDA ya msingi (mgawanyo wa mikoa, mada, urefu wa maneno, vocabulary)
- Umekagua na kusafisha data leakage kati ya splits — hatua muhimu sana kabla
  ya kupima ubora wa model yoyote
- Data iko tayari kwenye `./prepared_dataset/` — JSONL na HuggingFace `datasets` format
- Una msingi (skeleton) wa LoRA fine-tuning wa kuanzia

Kadiri dataset yako inavyokua (kuelekea lengo la maneno milioni 1), rudia hatua
za (1)-(3) mara kwa mara kabla ya fine-tune yoyote mpya.